In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pandas as pd

eeg_path = "/content/drive/MyDrive/data_n_back_test/eeg/eeg.parquet"

In [ ]:
df = pd.read_parquet(eeg_path)

In [ ]:
df = df[df["phase"] == 2].copy()

In [ ]:
eeg_cols = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

In [ ]:
WIN = 512
STRIDE = 512   # you can switch to 512 if you want no overlap

X_windows = []
y_windows = []
subjects = []

for (subj, test), g in df.groupby(["subject", "test"], sort=False):

    if "timestamp" in df.columns and "counter" in df.columns:
        g = g.sort_values(["timestamp", "counter"], kind="mergesort")
    elif "timestamp" in df.columns:
        g = g.sort_values("timestamp", kind="mergesort")
    else:
        g = g.sort_index()

    X = g[eeg_cols].to_numpy(dtype=np.float32)
    y = g["test"].to_numpy()

    if len(X) < WIN:
        continue

    for start in range(0, len(X) - WIN + 1, STRIDE):
        end = start + WIN
        X_windows.append(X[start:end].T)
        y_windows.append(y[start])
        subjects.append(subj)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows)
subjects  = np.array(subjects)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("subjects :", subjects.shape)

X_windows: (15007, 14, 512)
y_windows: (15007,)
subjects : (15007,)


In [ ]:
import numpy as np

print("Starting bandpower computation")
print("X_windows dtype:", X_windows.dtype)
print("X_windows shape:", X_windows.shape)

FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N, C, T = X_windows.shape
print("N:", N, "C:", C, "T:", T)

# ---- FFT frequencies ----
freqs = np.fft.rfftfreq(T, d=1.0/FS)
print("freqs shape:", freqs.shape)
print("First 10 freqs:", freqs[:10])
print("Last 10 freqs:", freqs[-10:])

# ---- FFT ----
X_fft = np.fft.rfft(X_windows, axis=-1)
print("X_fft shape:", X_fft.shape)
print("X_fft dtype:", X_fft.dtype)

# ---- PSD ----
PSD = (np.abs(X_fft) ** 2) / T
print("PSD shape:", PSD.shape)
print("PSD min/max:", PSD.min(), PSD.max())

# ---- Band extraction ----
bp_list = []

for name, f_lo, f_hi in bands:
    mask = (freqs >= f_lo) & (freqs < f_hi)

    print("\nBand:", name)
    print("Freq range:", f_lo, "-", f_hi)
    print("Number of bins in mask:", mask.sum())

    bp = PSD[..., mask]
    print("PSD slice shape:", bp.shape)

    bp_mean = bp.mean(axis=-1)
    print("After mean over freq axis:", bp_mean.shape)

    bp_list.append(bp_mean)

# ---- Stack bands ----
X_bp = np.stack(bp_list, axis=-1)
print("\nFinal stacked bandpower shape:", X_bp.shape)

# ---- Flatten ----
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)
print("Flattened shape:", X_bp_flat.shape)

print("Done bandpower.")


Starting bandpower computation
X_windows dtype: float32
X_windows shape: (15007, 14, 512)
N: 15007 C: 14 T: 512
freqs shape: (257,)
First 10 freqs: [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25]
Last 10 freqs: [61.75 62.   62.25 62.5  62.75 63.   63.25 63.5  63.75 64.  ]
X_fft shape: (15007, 14, 257)
X_fft dtype: complex64
PSD shape: (15007, 14, 257)
PSD min/max: 0.0 13989107000.0

Band: theta
Freq range: 4 - 8
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: alpha
Freq range: 8 - 12
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: beta
Freq range: 12 - 30
Number of bins in mask: 72
PSD slice shape: (15007, 14, 72)
After mean over freq axis: (15007, 14)

Final stacked bandpower shape: (15007, 14, 3)
Flattened shape: (15007, 42)
Done bandpower.


In [ ]:
# X_bp currently shape: (15007, 14, 3)

print("Before relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Relative within each channel
X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)

print("After relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Flatten
N = X_bp.shape[0]
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

print("X_bp_flat shape:", X_bp_flat.shape)

Before relative normalization:
Example band sums (first window, first channel): 7497.5425
After relative normalization:
Example band sums (first window, first channel): 1.0
X_bp_flat shape: (15007, 42)


In [ ]:
import torch

X_time_t = torch.tensor(X_windows, dtype=torch.float32)
X_bp_t   = torch.tensor(X_bp_flat, dtype=torch.float32)

# Encode labels 0..2
unique_labels = np.unique(y_windows)
label_map = {v:i for i,v in enumerate(unique_labels)}
y_encoded = np.vectorize(label_map.get)(y_windows).astype(np.int64)

y_t = torch.tensor(y_encoded, dtype=torch.long)

print("X_time_t:", X_time_t.shape)
print("X_bp_t:", X_bp_t.shape)
print("y_t:", y_t.shape)
print("Label mapping:", label_map)

X_time_t: torch.Size([15007, 14, 512])
X_bp_t: torch.Size([15007, 42])
y_t: torch.Size([15007])
Label mapping: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2}


In [ ]:

N, C, T = X_windows.shape

cov_feats = []

for i in range(N):
    X = X_windows[i]                # (14, 512)
    X = X - X.mean(axis=1, keepdims=True)  # zero-mean per channel
    cov = (X @ X.T) / (T - 1)      # (14,14)

    # take upper triangle including diagonal
    iu = np.triu_indices(C)
    cov_vec = cov[iu]              # 105 features
    cov_feats.append(cov_vec)

X_cov = np.array(cov_feats, dtype=np.float32)

print("X_cov shape:", X_cov.shape)  # should be (N, 105)

X_cov shape: (15007, 105)


In [ ]:
X_cov_t = torch.tensor(X_cov, dtype=torch.float32)

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

# Bandpass filter design
b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

N, C, T = X_windows.shape
plv_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter each channel
    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C)])

    # Hilbert transform → analytic signal
    analytic = hilbert(X_filt)
    phase = np.angle(analytic)  # (14, T)

    # Compute PLV matrix
    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    # Upper triangle including diagonal
    iu = np.triu_indices(C)
    plv_vec = plv_mat[iu]  # 105 dims
    plv_feats.append(plv_vec)

X_plv = np.array(plv_feats, dtype=np.float32)
print("X_plv shape:", X_plv.shape)  # (N, 105)

X_plv_t = torch.tensor(X_plv, dtype=torch.float32)

X_plv shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Alpha (8–12 Hz) PLV
# ----------------------------------------

low_a, high_a = 8, 12
b_a, a_a = butter(4, [low_a/(fs/2), high_a/(fs/2)], btype='band')

plv_alpha_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter
    X_filt = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C)])

    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C)
    plv_alpha_feats.append(plv_mat[iu])

X_plv_alpha = np.array(plv_alpha_feats, dtype=np.float32)
print("Alpha PLV shape:", X_plv_alpha.shape)

Alpha PLV shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Combine Theta + Alpha PLV
# ----------------------------------------

X_plv_combined = np.concatenate([X_plv, X_plv_alpha], axis=1)
print("Combined PLV shape:", X_plv_combined.shape)  # (N, 210)

X_plv_t = torch.tensor(X_plv_combined, dtype=torch.float32)

Combined PLV shape: (15007, 210)


In [ ]:
# Subjects to remove
bad_subjects = ["subject_05", "subject_11", "subject_16"]

print("Original total windows:", len(subjects))

mask = ~np.isin(subjects, bad_subjects)

X_time_t = X_time_t[mask]
X_bp_t   = X_bp_t[mask]
y_t      = y_t[mask]
subjects = subjects[mask]
X_cov_t  = X_cov_t[mask]   # ADD THIS
X_plv_t  = X_plv_t[mask]   # ADD THIS

print("After removal total windows:", len(subjects))
print("Remaining unique subjects:", np.unique(subjects))
print("Remaining count:", len(np.unique(subjects)))

Original total windows: 15007
After removal total windows: 12159
Remaining unique subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']
Remaining count: 13


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([12159, 105]) torch.Size([12159, 210]) torch.Size([12159])


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([12159, 105]) torch.Size([12159, 210]) torch.Size([12159])


In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            db      = db.to(device)

            # SAFE: supports both versions of model
            try:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi, db)
            except TypeError:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

In [ ]:
import torch

def build_roi_workload_features_from_bp(
    X_bp_flat: torch.Tensor,          # (N, 42)
    n_ch: int = 14,
    bands=("theta", "alpha", "beta"),  # must match how you built X_bp
    frontal_idx=(0, 1, 2, 3),          # CHANGE THIS to match your montage
    parietal_idx=(10, 11, 12, 13),     # CHANGE THIS to match your montage
    use_log: bool = True,
    use_relative: bool = False,        # relative bandpower within channel (theta/alpha/beta)
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Returns ROI features (N, 3):
      [frontal_theta, parietal_alpha, theta_alpha_ratio]
    """
    assert X_bp_flat.ndim == 2, "X_bp_flat must be (N, 42)"
    n_bands = len(bands)
    assert X_bp_flat.shape[1] == n_ch * n_bands, "bp dim mismatch"

    # reshape -> (N, ch, band)
    X = X_bp_flat.view(-1, n_ch, n_bands)

    # optional relative power within channel
    if use_relative:
        denom = X.sum(dim=2, keepdim=True).clamp_min(eps)
        X = X / denom

    # optional log (common for EEG power)
    if use_log:
        X = torch.log(X.clamp_min(eps))

    # band indices
    band_to_i = {b: i for i, b in enumerate(bands)}
    theta_i = band_to_i["theta"]
    alpha_i = band_to_i["alpha"]

    theta_ch = X[:, :, theta_i]  # (N, ch)
    alpha_ch = X[:, :, alpha_i]  # (N, ch)

    frontal_theta = theta_ch[:, list(frontal_idx)].mean(dim=1)   # (N,)
    parietal_alpha = alpha_ch[:, list(parietal_idx)].mean(dim=1) # (N,)

    # ratio: theta_frontal / alpha_parietal
    # If using log-space, ratio becomes (log theta - log alpha) which is great + stable.
    if use_log:
        theta_alpha_ratio = frontal_theta - parietal_alpha
    else:
        theta_alpha_ratio = frontal_theta / parietal_alpha.clamp_min(eps)

    roi = torch.stack([frontal_theta, parietal_alpha, theta_alpha_ratio], dim=1)  # (N,3)
    return roi

In [ ]:
def extract_embeddings(model, loader, device):

    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)

    return E, Y

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
import numpy as np
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )

In [ ]:

from torch.utils.data import Sampler
import random
from collections import defaultdict

class SubjectBalancedBatchSampler(Sampler):
    """
    Creates batches containing samples from multiple subjects.
    Works with your existing EEGDataset (which has .subj tensor).
    """

    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        # Build index list per subject
        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())

        # samples per subject inside batch
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        # Shuffle indices per subject
        for s in self.subjects:
            random.shuffle(self.subj_to_indices[s])

        # Create subject iterators
        subj_iters = {s: iter(self.subj_to_indices[s]) for s in self.subjects}

        while True:
            # randomly choose subjects for this batch
            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)

            batch = []

            for s in chosen_subjects:
                for _ in range(self.samples_per_subject):
                    try:
                        batch.append(next(subj_iters[s]))
                    except StopIteration:
                        return  # stop when any subject runs out

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        # approximate number of batches
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return (min_samples // self.samples_per_subject)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5920 | best 0.5920
Epoch 02 | zero-shot acc 0.6257 | best 0.6257
Epoch 03 | zero-shot acc 0.6025 | best 0.6257
Epoch 04 | zero-shot acc 0.6383 | best 0.6383
Epoch 05 | zero-shot acc 0.6267 | best 0.6383
Epoch 06 | zero-shot acc 0.6288 | best 0.6383
Epoch 07 | zero-shot acc 0.6456 | best 0.6456
Epoch 08 | zero-shot acc 0.6278 | best 0.6456
Epoch 09 | zero-shot acc 0.6288 | best 0.6456
Epoch 10 | zero-shot acc 0.6109 | best 0.6456
AdaBN + Mahalanobis few-shot acc: 0.7014590501785278

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5080 | best 0.5080
Epoch 02 | zero-shot acc 0.5602 | best 0.5602
Epoch 03 | zero-shot acc 0.6251 | best 0.6251
Epoch 04 | zero-shot acc 0.6017 | best 0.6251
Epoch 05 | zero-shot acc 0.5421 | best 0.6251
Epoch 06 | zero-shot acc 0.5804 | best 0.6251
Epoch 07 | zero-shot acc 0.5389 | best 0.6251
Epoch 08 | zero-shot acc 0.5186 | best 0.6251
Epoch 09 | zero-shot acc 0.5218 | best 0.

In [ ]:
import torch

# =========================
# VERIFY SHAPES FIRST (IMPORTANT)
# =========================
print("X_time:", X_time_t.shape)
print("X_bp:  ", X_bp_t.shape)
print("X_cov: ", X_cov_t.shape)
print("X_plv: ", X_plv_t.shape)
print("y:     ", y_t.shape)
print("subj:  ", len(subjects))

assert len(X_time_t) == len(X_bp_t) == len(X_cov_t) == len(X_plv_t) == len(y_t) == len(subjects), "Mismatch!"

# =========================
# BUILD DICT
# =========================
data_nb = {
    "X_time": X_time_t,
    "X_bp":   X_bp_t,
    "X_cov":  X_cov_t,
    "X_plv":  X_plv_t,
    "y":      y_t,
    "subjects": subjects
}

# =========================
# SAVE
# =========================
save_path = "/content/drive/MyDrive/EEG_eeg_preprocessed_nback.pt"

torch.save(data_nb, save_path)

print(f"Saved to: {save_path}")

X_time: torch.Size([12159, 14, 512])
X_bp:   torch.Size([12159, 42])
X_cov:  torch.Size([12159, 105])
X_plv:  torch.Size([12159, 210])
y:      torch.Size([12159])
subj:   12159
Saved to: /content/drive/MyDrive/EEG_eeg_preprocessed_nback.pt


In [ ]:
eeg_cols = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

In [ ]:
# Example: AF3, F3, AF4
KEEP_IDXS = [0, 2, 13]

In [ ]:
def select_channels(X_time, keep_idxs):
    return X_time[:, keep_idxs, :]

In [ ]:
class FusionEncoder3Ch(FusionEncoder):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__(emb_dim, bp_dim, roi_dim)

        # override first conv layer (14 → 3)
        self.time_branch[0] = nn.Conv1d(
            3, 3, kernel_size=7, padding=3, groups=3
        )

In [ ]:
def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 1.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)


# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER
# ============================================================

teacher = teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),   # zeroed out like in doc 2
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6477 | best 0.6477
Epoch 02 | zero-shot acc 0.6656 | best 0.6656
Epoch 03 | zero-shot acc 0.6688 | best 0.6688
Epoch 04 | zero-shot acc 0.6772 | best 0.6772
Epoch 05 | zero-shot acc 0.7003 | best 0.7003
Epoch 06 | zero-shot acc 0.6814 | best 0.7003
Epoch 07 | zero-shot acc 0.6772 | best 0.7003
Epoch 08 | zero-shot acc 0.6845 | best 0.7003
Epoch 09 | zero-shot acc 0.6751 | best 0.7003
Epoch 10 | zero-shot acc 0.6803 | best 0.7003
AdaBN + Mahalanobis few-shot acc: 0.7396184206008911

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3504 | best 0.3504
Epoch 02 | zero-shot acc 0.3663 | best 0.3663
Epoch 03 | zero-shot acc 0.3642 | best 0.3663
Epoch 04 | zero-shot acc 0.3962 | best 0.3962
Epoch 05 | zero-shot acc 0.4324 | best 0.4324
Epoch 06 | zero-shot acc 0.4

KeyboardInterrupt: 

In [ ]:
TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []
global_best_acc = -1.0
global_best_state = None

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        # track global best across all folds
        if acc > global_best_acc:
            global_best_acc = acc
            global_best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | global best {global_best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


# =========================
# SAVE GLOBAL BEST AS TEACHER
# =========================
torch.save({"model_state_dict": global_best_state}, TEACHER_CKPT_PATH)
print(f"\nTeacher saved to: {TEACHER_CKPT_PATH} (global best zero-shot acc: {global_best_acc:.4f})")

print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6057 | best 0.6057 | global best 0.6057
Epoch 02 | zero-shot acc 0.5920 | best 0.6057 | global best 0.6057
Epoch 03 | zero-shot acc 0.6172 | best 0.6172 | global best 0.6172
Epoch 04 | zero-shot acc 0.6141 | best 0.6172 | global best 0.6172
Epoch 05 | zero-shot acc 0.6120 | best 0.6172 | global best 0.6172
Epoch 06 | zero-shot acc 0.6299 | best 0.6299 | global best 0.6299
Epoch 07 | zero-shot acc 0.6288 | best 0.6299 | global best 0.6299
Epoch 08 | zero-shot acc 0.6162 | best 0.6299 | global best 0.6299
Epoch 09 | zero-shot acc 0.6109 | best 0.6299 | global best 0.6299
Epoch 10 | zero-shot acc 0.6099 | best 0.6299 | global best 0.6299
AdaBN + Mahalanobis few-shot acc: 0.7104377150535583

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5389 | best 0.5389 | global best 0.6299
Epoch 02 | zero-shot acc 0.5527 | best 0.5527 | global best 0.6299
Epoch 03 | zero-shot acc 0.6305 | best 0.6305 | global best 0.630

In [ ]:
def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 1.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5762 | best 0.5762
Epoch 02 | zero-shot acc 0.6015 | best 0.6015
Epoch 03 | zero-shot acc 0.5994 | best 0.6015
Epoch 04 | zero-shot acc 0.6236 | best 0.6236
Epoch 05 | zero-shot acc 0.6383 | best 0.6383
Epoch 06 | zero-shot acc 0.6730 | best 0.6730
Epoch 07 | zero-shot acc 0.6425 | best 0.6730
Epoch 08 | zero-shot acc 0.6677 | best 0.6730
Epoch 09 | zero-shot acc 0.6772 | best 0.6772
Epoch 10 | zero-shot acc 0.7003 | best 0.7003
AdaBN + Mahalanobis few-shot acc: 0.8260381817817688

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3983 | best 0.3983
Epoch 02 | zero-shot acc 0.3663 | best 0.3983
Epoch 03 | zero-shot acc 0.3685 | best 0.3983
Epoch 04 | zero-shot acc 0.3813 | best 0.3983
Epoch 05 | zero-shot acc 0.4036 | best 0.4036
Epoch 06 | zero-shot acc 0.4

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 0.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6488 | best 0.6488
Epoch 02 | zero-shot acc 0.6593 | best 0.6593
Epoch 03 | zero-shot acc 0.6814 | best 0.6814
Epoch 04 | zero-shot acc 0.6951 | best 0.6951
Epoch 05 | zero-shot acc 0.6898 | best 0.6951
Epoch 06 | zero-shot acc 0.6845 | best 0.6951
Epoch 07 | zero-shot acc 0.6814 | best 0.6951
Epoch 08 | zero-shot acc 0.6940 | best 0.6951
Epoch 09 | zero-shot acc 0.7140 | best 0.7140
Epoch 10 | zero-shot acc 0.6951 | best 0.7140
AdaBN + Mahalanobis few-shot acc: 0.760942816734314

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3908 | best 0.3908
Epoch 02 | zero-shot acc 0.3461 | best 0.3908
Epoch 03 | zero-shot acc 0.3919 | best 0.3919
Epoch 04 | zero-shot acc 0.3855 | best 0.3919
Epoch 05 | zero-shot acc 0.3876 | best 0.3919
Epoch 06 | zero-shot acc 0.45

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 0.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

# Experiment 1: pick 3 BAD channels (e.g. occipital)
KEEP_IDXS = [6, 7, 5]   # O1, O2, P7
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5089 | best 0.5089
Epoch 02 | zero-shot acc 0.5026 | best 0.5089
Epoch 03 | zero-shot acc 0.5247 | best 0.5247
Epoch 04 | zero-shot acc 0.5342 | best 0.5342
Epoch 05 | zero-shot acc 0.5321 | best 0.5342
Epoch 06 | zero-shot acc 0.5373 | best 0.5373
Epoch 07 | zero-shot acc 0.5657 | best 0.5657
Epoch 08 | zero-shot acc 0.5542 | best 0.5657
Epoch 09 | zero-shot acc 0.5489 | best 0.5657
Epoch 10 | zero-shot acc 0.5689 | best 0.5689
AdaBN + Mahalanobis few-shot acc: 0.6879910230636597

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4643 | best 0.4643
Epoch 02 | zero-shot acc 0.4601 | best 0.4643
Epoch 03 | zero-shot acc 0.4931 | best 0.4931
Epoch 04 | zero-shot acc 0.4750 | best 0.4931
Epoch 05 | zero-shot acc 0.4675 | best 0.4931
Epoch 06 | zero-shot acc 0.4

In [ ]:
print(f"Train subjects: {np.unique(subj_train)}")
print(f"Test subject: {test_subj}")

Train subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14']
Test subject: subject_15


In [ ]:
# Run this right after the support_mask is built in the few-shot section
# to verify no support samples appear in the query set

print("=== VERIFYING SUPPORT/QUERY SPLIT ===")
print(f"Total test samples:   {len(all_y)}")
print(f"Support samples:      {support_mask.sum().item()}  ({100*support_mask.float().mean().item():.1f}%)")
print(f"Query samples:        {(~support_mask).sum().item()}  ({100*(~support_mask).float().mean().item():.1f}%)")

# verify no overlap
overlap = support_mask.sum().item() + (~support_mask).sum().item()
assert overlap == len(all_y), "OVERLAP DETECTED — support and query are not disjoint!"
print(f"Total accounted for:  {overlap} ✓ no overlap")

# verify per-class support counts
print("\nPer-class support counts:")
for c in torch.unique(all_y):
    class_total = (all_y == c).sum().item()
    class_support = (all_y[support_mask] == c).sum().item()
    class_query = (all_y[~support_mask] == c).sum().item()
    print(f"  Class {c.item()}: total={class_total}, support={class_support}, query={class_query}")

# verify evaluation is only on query
print(f"\nFew-shot acc computed on {(~support_mask).sum().item()} query samples only ✓")

=== VERIFYING SUPPORT/QUERY SPLIT ===
Total test samples:   913
Support samples:      60  (6.6%)
Query samples:        853  (93.4%)
Total accounted for:  913 ✓ no overlap

Per-class support counts:
  Class 0: total=310, support=20, query=290
  Class 1: total=300, support=20, query=280
  Class 2: total=303, support=20, query=283

Few-shot acc computed on 853 query samples only ✓


In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 1.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

# Experiment 1: pick 3 BAD channels (e.g. occipital)
KEEP_IDXS = [6, 7, 5]   # O1, O2, P7
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.4963 | best 0.4963
Epoch 02 | zero-shot acc 0.4932 | best 0.4963
Epoch 03 | zero-shot acc 0.5110 | best 0.5110
Epoch 04 | zero-shot acc 0.5100 | best 0.5110
Epoch 05 | zero-shot acc 0.5268 | best 0.5268
Epoch 06 | zero-shot acc 0.5237 | best 0.5268
Epoch 07 | zero-shot acc 0.5352 | best 0.5352
Epoch 08 | zero-shot acc 0.5710 | best 0.5710
Epoch 09 | zero-shot acc 0.5910 | best 0.5910
Epoch 10 | zero-shot acc 0.6067 | best 0.6067
AdaBN + Mahalanobis few-shot acc: 0.7463524341583252

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4302 | best 0.4302
Epoch 02 | zero-shot acc 0.4271 | best 0.4302
Epoch 03 | zero-shot acc 0.4760 | best 0.4760
Epoch 04 | zero-shot acc 0.3983 | best 0.4760
Epoch 05 | zero-shot acc 0.4643 | best 0.4760
Epoch 06 | zero-shot acc 0.4

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 0.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )
    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        # --- ABLATION: comment one out at a time ---
        z_time = torch.zeros_like(z_time)  # kill time branch → only bp
        # z_bp = torch.zeros_like(z_bp)    # kill bp branch → only time

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6593 | best 0.6593
Epoch 02 | zero-shot acc 0.6109 | best 0.6593
Epoch 03 | zero-shot acc 0.6393 | best 0.6593
Epoch 04 | zero-shot acc 0.6488 | best 0.6593
Epoch 05 | zero-shot acc 0.6719 | best 0.6719
Epoch 06 | zero-shot acc 0.6698 | best 0.6719
Epoch 07 | zero-shot acc 0.6635 | best 0.6719
Epoch 08 | zero-shot acc 0.6393 | best 0.6719
Epoch 09 | zero-shot acc 0.6709 | best 0.6719
Epoch 10 | zero-shot acc 0.6530 | best 0.6719
AdaBN + Mahalanobis few-shot acc: 0.7384961247444153

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3184 | best 0.3184
Epoch 02 | zero-shot acc 0.3174 | best 0.3184
Epoch 03 | zero-shot acc 0.3259 | best 0.3259
Epoch 04 | zero-shot acc 0.3248 | best 0.3259
Epoch 05 | zero-shot acc 0.3280 | best 0.3280
Epoch 06 | zero-shot acc 0.3

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 1.0

# choose 3 channels from your order:
# 0 EEG.AF3
# 1 EEG.F7
# 2 EEG.F3
# 3 EEG.FC5
# 4 EEG.T7
# 5 EEG.P7
# 6 EEG.O1
# 7 EEG.O2
# 8 EEG.P8
# 9 EEG.T8
# 10 EEG.FC6
# 11 EEG.F4
# 12 EEG.F8
# 13 EEG.AF4

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    """
    Same idea as your original sampler.
    """
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    """
    X_time: (N, C, T)
    returns: (N, C*len(bands)) flattened relative bandpower
    """
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)   # (N, C)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)       # (N, C, 3)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)

# ============================================================
# TEACHER MODEL (14-CHANNEL ORIGINAL STYLE, TIME + BP ONLY)
# ============================================================

class FusionEncoderTeacher(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        if x_roi is None:
            x_roi = torch.zeros((x_bp.shape[0], 0), device=x_bp.device, dtype=x_bp.dtype)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveTeacherModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderTeacher(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov=None, x_plv=None, x_roi=None):
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )
    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        # --- ABLATION: comment one out at a time ---
        z_time = torch.zeros_like(z_time)  # kill time branch → only bp
        # z_bp = torch.zeros_like(z_bp)    # kill bp branch → only time

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER (use the same ContrastiveModel from doc 2)
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)   # should be (N, 9)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    # student inputs
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    # teacher inputs (full 14-ch)
    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    # student time normalization
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # student bp normalization
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # teacher normalization (match teacher training style)
    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    X_roi_train_teacher = torch.zeros((X_bp_train_teacher.shape[0], 0), dtype=X_bp_train_teacher.dtype)
    X_roi_test_teacher  = torch.zeros((X_bp_test_teacher.shape[0], 0), dtype=X_bp_test_teacher.dtype)

    # =========================
    # PRECOMPUTE TEACHER TARGET EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
    teacher_model=teacher,
    X_time=X_time_train_teacher,
    X_bp=X_bp_train_teacher,
    X_cov=torch.zeros_like(X_cov_t[train_idx]),
    X_plv=X_plv_t[train_idx].clone(),
    device=device,
    batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = F.mse_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best student before adaptation
    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    # normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6614 | best 0.6614
Epoch 02 | zero-shot acc 0.6562 | best 0.6614
Epoch 03 | zero-shot acc 0.6383 | best 0.6614
Epoch 04 | zero-shot acc 0.6509 | best 0.6614
Epoch 05 | zero-shot acc 0.6625 | best 0.6625
Epoch 06 | zero-shot acc 0.6656 | best 0.6656
Epoch 07 | zero-shot acc 0.6635 | best 0.6656
Epoch 08 | zero-shot acc 0.6572 | best 0.6656
Epoch 09 | zero-shot acc 0.6719 | best 0.6719
Epoch 10 | zero-shot acc 0.6646 | best 0.6719
AdaBN + Mahalanobis few-shot acc: 0.7272727489471436

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3259 | best 0.3259
Epoch 02 | zero-shot acc 0.3323 | best 0.3323
Epoch 03 | zero-shot acc 0.3174 | best 0.3323
Epoch 04 | zero-shot acc 0.3184 | best 0.3323
Epoch 05 | zero-shot acc 0.3227 | best 0.3323
Epoch 06 | zero-shot acc 0.3

In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH 14-CHANNEL TEACHER GUIDANCE
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

# distillation strength
DISTILL_LAMBDA = 0.5

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def relational_distill_loss(e_student, e_teacher):
    """
    Match pairwise similarity structure between student and teacher.
    Student doesn't need to copy exact values, just agree on which
    samples are similar/different — works across different input spaces.
    """
    s_sim = F.normalize(e_student, dim=1) @ F.normalize(e_student, dim=1).T
    t_sim = F.normalize(e_teacher, dim=1) @ F.normalize(e_teacher, dim=1).T
    return F.mse_loss(s_sim, t_sim)


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)


# ============================================================
# STUDENT MODEL (3-CHANNEL TIME + 3-CHANNEL BP ONLY)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    # =========================
    # PRECOMPUTE TEACHER EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_train_teacher,
        X_bp=X_bp_train_teacher,
        X_cov=torch.zeros_like(X_cov_t[train_idx]),
        X_plv=X_plv_t[train_idx].clone(),
        device=device,
        batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb_np = train_ds.subj[batch_indices]
            sb = sb_np.to(device)

            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = relational_distill_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()

    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[support_mask].to(device)
    y_support = all_y[support_mask].to(device)
    e_query   = all_e[~support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor([label_map[int(v.item())] for v in y_query], device=device)

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

print("\n3-CHANNEL STUDENT + TEACHER DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6414 | best 0.6414
Epoch 02 | zero-shot acc 0.6498 | best 0.6498
Epoch 03 | zero-shot acc 0.6845 | best 0.6845
Epoch 04 | zero-shot acc 0.6961 | best 0.6961
Epoch 05 | zero-shot acc 0.6835 | best 0.6961
Epoch 06 | zero-shot acc 0.6803 | best 0.6961
Epoch 07 | zero-shot acc 0.6719 | best 0.6961
Epoch 08 | zero-shot acc 0.6866 | best 0.6961
Epoch 09 | zero-shot acc 0.7129 | best 0.7129
Epoch 10 | zero-shot acc 0.6993 | best 0.7129
AdaBN + Mahalanobis few-shot acc: 0.7867565155029297

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3610 | best 0.3610
Epoch 02 | zero-shot acc 0.4196 | best 0.4196
Epoch 03 | zero-shot acc 0.3610 | best 0.4196
Epoch 04 | zero-shot acc 0.3962 | best 0.4196
Epoch 05 | zero-shot acc 0.4260 | best 0.4260
Epoch 06 | zero-shot acc 0.4

In [ ]:
!pip install higher


In [ ]:
import higher

In [ ]:
# pip install higher   ← run this once first

import higher
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS           = 10
BATCH            = 396
LR               = 1e-3
LAMBDA           = 0.5
TEMP             = 0.1
PROTO_LAMBDA     = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK           = 0.4

INNER_LR    = 0.01   # how fast to adapt during the inner step
INNER_STEPS = 1      # 1 gradient step is standard MAML
N_EPISODES  = 50     # how many MAML episodes per epoch

overall_results = []
global_best_acc = -1.0
global_best_state = None

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    # precompute subject index lookup for episodes
    train_subj_tensor = train_ds.subj  # (N,) int tensor
    train_subjects    = torch.unique(train_subj_tensor)

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        # ── MAML EPISODES ──────────────────────────────────────────
        for episode in range(N_EPISODES):

            # 1. pick one training subject to pretend is a stranger
            chosen_subj = train_subjects[
                torch.randint(len(train_subjects), (1,))
            ].item()

            subj_mask = (train_subj_tensor == chosen_subj)
            subj_idx  = subj_mask.nonzero(as_tuple=True)[0]  # indices into train arrays

            # need at least support + 1 query per class
            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # 2. split into support (20 per class) and query (the rest)
            support_idx, query_idx = [], []
            skip = False
            for c in classes_in_subj:
                c_mask  = (y_train[subj_idx] == c)
                c_idx   = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx = torch.cat(support_idx)
            query_idx   = torch.cat(query_idx)

            # 3. gather tensors
            xs_time = X_time_train[support_idx].to(device)
            xs_bp   = X_bp_train[support_idx].to(device)
            xs_cov  = X_cov_train[support_idx].to(device)
            xs_plv  = X_plv_train[support_idx].to(device)
            xs_roi  = X_roi_train[support_idx].to(device)
            ys      = y_train[support_idx].to(device)

            xq_time = X_time_train[query_idx].to(device)
            xq_bp   = X_bp_train[query_idx].to(device)
            xq_cov  = X_cov_train[query_idx].to(device)
            xq_plv  = X_plv_train[query_idx].to(device)
            xq_roi  = X_roi_train[query_idx].to(device)
            yq      = y_train[query_idx].to(device)

            # 4. MAML inner/outer loop
            inner_opt = torch.optim.SGD(model.parameters(), lr=INNER_LR)

            with higher.innerloop_ctx(
                model, inner_opt, copy_initial_weights=True
            ) as (fmodel, diffopt):

                # ── inner loop: adapt to support ──
                for _ in range(INNER_STEPS):
                    e_s, proj_s = fmodel(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
                    loss_inner  = (
                        prototype_ce_loss(e_s, ys, temperature=0.1)
                        + PROTO_LAMBDA * prototype_loss(e_s, ys)
                    )
                    diffopt.step(loss_inner)

                # ── outer loop: grade on query ──
                e_q, proj_q = fmodel(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)
                sb_q = train_ds.subj[query_idx].to(device)

                loss_outer = (
                    prototype_ce_loss(e_q, yq, temperature=0.1)
                    + LAMBDA       * supcon_diff_subject_loss(proj_q, yq, sb_q, temperature=TEMP)
                    + PROTO_LAMBDA * prototype_loss(e_q, yq)
                )

            opt.zero_grad()
            loss_outer.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (unchanged)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        if acc > global_best_acc:
            global_best_acc   = acc
            global_best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | global best {global_best_acc:.4f}")

    # load best before few-shot
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (COMPLETELY UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(xb_time.to(device), xb_bp.to(device),
                      xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device))

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs      = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists      = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred       = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

# =========================
# SAVE GLOBAL BEST AS TEACHER
# =========================
torch.save({"model_state_dict": global_best_state}, TEACHER_CKPT_PATH)
print(f"\nTeacher saved to: {TEACHER_CKPT_PATH} (global best zero-shot acc: {global_best_acc:.4f})")

print("\nMAML + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 02 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 03 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 04 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 05 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 06 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 07 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 08 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 09 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
Epoch 10 | zero-shot acc 0.5752 | best 0.5752 | global best 0.5752
AdaBN + Mahalanobis few-shot acc: 0.7654321193695068

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3152 | best 0.3152 | global best 0.5752
Epoch 02 | zero-shot acc 0.3152 | best 0.3152 | global best 0.5752
Epoch 03 | zero-shot acc 0.3152 | best 0.3152 | global best 0.575

KeyboardInterrupt: 

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS             = 10
BATCH              = 396
LR                 = 1e-3
LAMBDA             = 0.5
TEMP               = 0.1
PROTO_LAMBDA       = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK             = 0.4
N_EPISODES         = 50  # episodic few-shot episodes per epoch

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    # precompute per-subject index lookup for episodes
    train_subj_tensor = train_ds.subj
    train_subjects    = torch.unique(train_subj_tensor).tolist()

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        # ── PART 1: regular contrastive training (same as before) ──
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # ── PART 2: episodic few-shot training ──
        # simulate the exact test scenario: 20 support per class -> classify query
        for episode in range(N_EPISODES):

            # pick a random training subject to treat as "unseen"
            chosen_subj = random.choice(train_subjects)
            subj_mask   = (train_subj_tensor == chosen_subj)
            subj_idx    = subj_mask.nonzero(as_tuple=True)[0]

            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # split into support and query
            support_idx_ep, query_idx_ep = [], []
            skip = False
            for c in classes_in_subj:
                c_mask = (y_train[subj_idx] == c)
                c_idx  = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx_ep.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx_ep.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx_ep = torch.cat(support_idx_ep)
            query_idx_ep   = torch.cat(query_idx_ep)

            # get support embeddings
            xs_time = X_time_train[support_idx_ep].to(device)
            xs_bp   = X_bp_train[support_idx_ep].to(device)
            xs_cov  = X_cov_train[support_idx_ep].to(device)
            xs_plv  = X_plv_train[support_idx_ep].to(device)
            xs_roi  = X_roi_train[support_idx_ep].to(device)
            ys      = y_train[support_idx_ep].to(device)

            # get query embeddings
            xq_time = X_time_train[query_idx_ep].to(device)
            xq_bp   = X_bp_train[query_idx_ep].to(device)
            xq_cov  = X_cov_train[query_idx_ep].to(device)
            xq_plv  = X_plv_train[query_idx_ep].to(device)
            xq_roi  = X_roi_train[query_idx_ep].to(device)
            yq      = y_train[query_idx_ep].to(device)

            model.train()
            e_support, _ = model(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
            e_query,   _ = model(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)

            # compute prototypes from support
            classes_ep = torch.unique(ys)
            protos_ep  = torch.stack([
                e_support[ys == c].mean(dim=0) for c in classes_ep
            ])

            # classify query using prototypes (cosine similarity)
            e_query_norm  = F.normalize(e_query,   dim=1)
            protos_norm   = F.normalize(protos_ep, dim=1)
            logits_ep     = e_query_norm @ protos_norm.T / 0.1

            # map query labels to local indices
            label_map_ep = {int(c.item()): i for i, c in enumerate(classes_ep)}
            yq_local     = torch.tensor(
                [label_map_ep[int(v.item())] for v in yq],
                device=device
            )

            loss_episode = F.cross_entropy(logits_ep, yq_local)

            opt.zero_grad()
            loss_episode.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(
                xb_time.to(device), xb_bp.to(device),
                xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device)
            )

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred  = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nEPISODIC + CONTRASTIVE LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

In [ ]:
# pip install higher   ← run this once first

import higher
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS           = 10
BATCH            = 396
LR               = 1e-3
LAMBDA           = 0.5
TEMP             = 0.1
PROTO_LAMBDA     = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK           = 0.4

INNER_LR    = 0.01   # how fast to adapt during the inner step
INNER_STEPS = 1      # 1 gradient step is standard MAML
N_EPISODES  = 50     # how many MAML episodes per epoch

overall_results = []
global_best_acc = -1.0
global_best_state = None

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    # precompute subject index lookup for episodes
    train_subj_tensor = train_ds.subj  # (N,) int tensor
    train_subjects    = torch.unique(train_subj_tensor)

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        # ── MAML EPISODES ──────────────────────────────────────────
        for episode in range(N_EPISODES):

            # 1. pick one training subject to pretend is a stranger
            chosen_subj = train_subjects[
                torch.randint(len(train_subjects), (1,))
            ].item()

            subj_mask = (train_subj_tensor == chosen_subj)
            subj_idx  = subj_mask.nonzero(as_tuple=True)[0]  # indices into train arrays

            # need at least support + 1 query per class
            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # 2. split into support (20 per class) and query (the rest)
            support_idx, query_idx = [], []
            skip = False
            for c in classes_in_subj:
                c_mask  = (y_train[subj_idx] == c)
                c_idx   = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx = torch.cat(support_idx)
            query_idx   = torch.cat(query_idx)

            # 3. gather tensors
            xs_time = X_time_train[support_idx].to(device)
            xs_bp   = X_bp_train[support_idx].to(device)
            xs_cov  = X_cov_train[support_idx].to(device)
            xs_plv  = X_plv_train[support_idx].to(device)
            xs_roi  = X_roi_train[support_idx].to(device)
            ys      = y_train[support_idx].to(device)

            xq_time = X_time_train[query_idx].to(device)
            xq_bp   = X_bp_train[query_idx].to(device)
            xq_cov  = X_cov_train[query_idx].to(device)
            xq_plv  = X_plv_train[query_idx].to(device)
            xq_roi  = X_roi_train[query_idx].to(device)
            yq      = y_train[query_idx].to(device)

            # 4. MAML inner/outer loop

            model.train()  # make sure model is in train mode for each episode
            inner_opt = torch.optim.SGD(model.parameters(), lr=INNER_LR)
            # inner_opt = torch.optim.SGD(model.parameters(), lr=INNER_LR)



            with higher.innerloop_ctx(model, inner_opt, copy_initial_weights=True, track_higher_grads=True) as (fmodel, diffopt):

              # inner loop
              for _ in range(INNER_STEPS):
                  e_s, proj_s = fmodel(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
                  loss_inner = prototype_ce_loss(e_s, ys, temperature=0.1)
                  diffopt.step(loss_inner)

              # outer loop — must be INSIDE the with block
              e_q, proj_q = fmodel(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)
              sb_q = train_ds.subj[query_idx].to(device)
              loss_outer = (
                  prototype_ce_loss(e_q, yq, temperature=0.1)
                  + LAMBDA * supcon_diff_subject_loss(proj_q, yq, sb_q, temperature=TEMP)
                  + PROTO_LAMBDA * prototype_loss(e_q, yq)
              )
              opt.zero_grad()
              loss_outer.backward()
              opt.step()  # ← moved inside the with block

        # =========================
        # ZERO-SHOT EVAL (unchanged)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        if acc > global_best_acc:
            global_best_acc   = acc
            global_best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | global best {global_best_acc:.4f}")

    # load best before few-shot
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (COMPLETELY UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(xb_time.to(device), xb_bp.to(device),
                      xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device))

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs      = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists      = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred       = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

# =========================
# SAVE GLOBAL BEST AS TEACHER
# =========================
torch.save({"model_state_dict": global_best_state}, TEACHER_CKPT_PATH)
print(f"\nTeacher saved to: {TEACHER_CKPT_PATH} (global best zero-shot acc: {global_best_acc:.4f})")

print("\nMAML + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 02 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 03 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 04 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 05 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 06 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 07 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331
Epoch 08 | zero-shot acc 0.5331 | best 0.5331 | global best 0.5331


KeyboardInterrupt: 

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS           = 10
BATCH            = 396
LR               = 1e-3
LAMBDA           = 0.5
TEMP             = 0.1
PROTO_LAMBDA     = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK           = 0.4

INNER_LR   = 0.01
INNER_STEPS = 1
N_EPISODES  = 50

overall_results = []
global_best_acc = -1.0
global_best_state = None

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    train_subj_tensor = train_ds.subj
    train_subjects    = torch.unique(train_subj_tensor)

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        for episode in range(N_EPISODES):

            # 1. pick a random training subject
            chosen_subj = train_subjects[torch.randint(len(train_subjects), (1,))].item()
            subj_mask   = (train_subj_tensor == chosen_subj)
            subj_idx    = subj_mask.nonzero(as_tuple=True)[0]

            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # 2. split support / query
            support_idx, query_idx = [], []
            skip = False
            for c in classes_in_subj:
                c_idx = subj_idx[(y_train[subj_idx] == c)]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx = torch.cat(support_idx)
            query_idx   = torch.cat(query_idx)

            # 3. gather tensors
            xs_time = X_time_train[support_idx].to(device)
            xs_bp   = X_bp_train[support_idx].to(device)
            xs_cov  = X_cov_train[support_idx].to(device)
            xs_plv  = X_plv_train[support_idx].to(device)
            xs_roi  = X_roi_train[support_idx].to(device)
            ys      = y_train[support_idx].to(device)

            xq_time = X_time_train[query_idx].to(device)
            xq_bp   = X_bp_train[query_idx].to(device)
            xq_cov  = X_cov_train[query_idx].to(device)
            xq_plv  = X_plv_train[query_idx].to(device)
            xq_roi  = X_roi_train[query_idx].to(device)
            yq      = y_train[query_idx].to(device)

            # 4. MANUAL MAML (no higher library)
            # 4. MANUAL MAML
            model.train()

            # save original weights
            original_state = copy.deepcopy(model.state_dict())

            # inner step: compute gradients on support
            e_s, _ = model(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
            loss_inner = prototype_ce_loss(e_s, ys, temperature=0.1)
            inner_grads = torch.autograd.grad(loss_inner, model.parameters(), allow_unused=True)

            # apply inner update
            with torch.no_grad():
                for p, g in zip(model.parameters(), inner_grads):
                    if g is not None:
                        p.data -= INNER_LR * g

            # outer step: compute query loss on adapted weights
            model.train()
            e_q, proj_q = model(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)
            sb_q = train_ds.subj[query_idx].to(device)
            loss_outer = (
                prototype_ce_loss(e_q, yq, temperature=0.1)
                + LAMBDA       * supcon_diff_subject_loss(proj_q, yq, sb_q, temperature=TEMP)
                + PROTO_LAMBDA * prototype_loss(e_q, yq)
            )

            # backward FIRST, then restore, then step
            opt.zero_grad()
            loss_outer.backward()          # ← before restore
            model.load_state_dict(original_state)  # ← restore after backward
            opt.step()                     # ← step on original weights with outer gradients

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        if acc > global_best_acc:
            global_best_acc   = acc
            global_best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f} | global best {global_best_acc:.4f}")

    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(xb_time.to(device), xb_bp.to(device),
                      xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device))

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs      = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists      = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred       = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))

torch.save({"model_state_dict": global_best_state}, TEACHER_CKPT_PATH)
print(f"\nTeacher saved to: {TEACHER_CKPT_PATH} (global best zero-shot acc: {global_best_acc:.4f})")

print("\nMAML + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5563 | best 0.5563 | global best 0.5563
Epoch 02 | zero-shot acc 0.6425 | best 0.6425 | global best 0.6425
Epoch 03 | zero-shot acc 0.6509 | best 0.6509 | global best 0.6509
Epoch 04 | zero-shot acc 0.6667 | best 0.6667 | global best 0.6667
Epoch 05 | zero-shot acc 0.6583 | best 0.6667 | global best 0.6667
Epoch 06 | zero-shot acc 0.6509 | best 0.6667 | global best 0.6667
Epoch 07 | zero-shot acc 0.6477 | best 0.6667 | global best 0.6667
Epoch 08 | zero-shot acc 0.6509 | best 0.6667 | global best 0.6667
Epoch 09 | zero-shot acc 0.6667 | best 0.6667 | global best 0.6667
Epoch 10 | zero-shot acc 0.6330 | best 0.6667 | global best 0.6667
AdaBN + Mahalanobis few-shot acc: 0.7856341600418091

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3813 | best 0.3813 | global best 0.6667
Epoch 02 | zero-shot acc 0.4547 | best 0.4547 | global best 0.6667
Epoch 03 | zero-shot acc 0.5048 | best 0.5048 | global best 0.666

NameError: name 'TEACHER_CKPT_PATH' is not defined

In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/EEG_eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")

Loaded N-back


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([12159, 105]) torch.Size([12159, 210]) torch.Size([12159])


In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS             = 10
BATCH              = 396
LR                 = 1e-3
LAMBDA             = 0.5
TEMP               = 0.1
PROTO_LAMBDA       = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK             = 0.4
N_EPISODES         = 50  # episodic few-shot episodes per epoch

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    # precompute per-subject index lookup for episodes
    train_subj_tensor = train_ds.subj
    train_subjects    = torch.unique(train_subj_tensor).tolist()

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        # ── PART 1: regular contrastive training (same as before) ──
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # ── PART 2: episodic few-shot training ──
        # simulate the exact test scenario: 20 support per class -> classify query
        for episode in range(N_EPISODES):

            # pick a random training subject to treat as "unseen"
            chosen_subj = random.choice(train_subjects)
            subj_mask   = (train_subj_tensor == chosen_subj)
            subj_idx    = subj_mask.nonzero(as_tuple=True)[0]

            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # split into support and query
            support_idx_ep, query_idx_ep = [], []
            skip = False
            for c in classes_in_subj:
                c_mask = (y_train[subj_idx] == c)
                c_idx  = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx_ep.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx_ep.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx_ep = torch.cat(support_idx_ep)
            query_idx_ep   = torch.cat(query_idx_ep)

            # get support embeddings
            xs_time = X_time_train[support_idx_ep].to(device)
            xs_bp   = X_bp_train[support_idx_ep].to(device)
            xs_cov  = X_cov_train[support_idx_ep].to(device)
            xs_plv  = X_plv_train[support_idx_ep].to(device)
            xs_roi  = X_roi_train[support_idx_ep].to(device)
            ys      = y_train[support_idx_ep].to(device)

            # get query embeddings
            xq_time = X_time_train[query_idx_ep].to(device)
            xq_bp   = X_bp_train[query_idx_ep].to(device)
            xq_cov  = X_cov_train[query_idx_ep].to(device)
            xq_plv  = X_plv_train[query_idx_ep].to(device)
            xq_roi  = X_roi_train[query_idx_ep].to(device)
            yq      = y_train[query_idx_ep].to(device)

            model.train()
            e_support, _ = model(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
            e_query,   _ = model(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)

            # compute prototypes from support
            classes_ep = torch.unique(ys)
            protos_ep  = torch.stack([
                e_support[ys == c].mean(dim=0) for c in classes_ep
            ])

            # classify query using prototypes (cosine similarity)
            e_query_norm  = F.normalize(e_query,   dim=1)
            protos_norm   = F.normalize(protos_ep, dim=1)
            logits_ep     = e_query_norm @ protos_norm.T / 0.1

            # map query labels to local indices
            label_map_ep = {int(c.item()): i for i, c in enumerate(classes_ep)}
            yq_local     = torch.tensor(
                [label_map_ep[int(v.item())] for v in yq],
                device=device
            )

            loss_episode = F.cross_entropy(logits_ep, yq_local)

            opt.zero_grad()
            loss_episode.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(
                xb_time.to(device), xb_bp.to(device),
                xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device)
            )

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred  = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nEPISODIC + CONTRASTIVE LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5868 | best 0.5868
Epoch 02 | zero-shot acc 0.6887 | best 0.6887
Epoch 03 | zero-shot acc 0.6236 | best 0.6887
Epoch 04 | zero-shot acc 0.7035 | best 0.7035
Epoch 05 | zero-shot acc 0.6740 | best 0.7035
Epoch 06 | zero-shot acc 0.6362 | best 0.7035
Epoch 07 | zero-shot acc 0.6362 | best 0.7035
Epoch 08 | zero-shot acc 0.6183 | best 0.7035
Epoch 09 | zero-shot acc 0.6698 | best 0.7035
Epoch 10 | zero-shot acc 0.7066 | best 0.7066
AdaBN + Mahalanobis few-shot acc: 0.8675645589828491

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.6784 | best 0.6784
Epoch 02 | zero-shot acc 0.6794 | best 0.6794
Epoch 03 | zero-shot acc 0.7114 | best 0.7114
Epoch 04 | zero-shot acc 0.5942 | best 0.7114
Epoch 05 | zero-shot acc 0.6464 | best 0.7114
Epoch 06 | zero-shot acc 0.5410 | best 0.7114
Epoch 07 | zero-shot acc 0.4313 | best 0.7114
Epoch 08 | zero-shot acc 0.5399 | best 0.7114
Epoch 09 | zero-shot acc 0.5410 | best 0.

In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS             = 15
BATCH              = 396
LR                 = 1e-3
LAMBDA             = 0.5
TEMP               = 0.1
PROTO_LAMBDA       = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK             = 0.4
N_EPISODES         = 100  # episodic few-shot episodes per epoch

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    # precompute per-subject index lookup for episodes
    train_subj_tensor = train_ds.subj
    train_subjects    = torch.unique(train_subj_tensor).tolist()

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        model.train()

        # ── PART 1: regular contrastive training (same as before) ──
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # ── PART 2: episodic few-shot training ──
        # simulate the exact test scenario: 20 support per class -> classify query
        for episode in range(N_EPISODES):

            # pick a random training subject to treat as "unseen"
            chosen_subj = random.choice(train_subjects)
            subj_mask   = (train_subj_tensor == chosen_subj)
            subj_idx    = subj_mask.nonzero(as_tuple=True)[0]

            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            # split into support and query
            support_idx_ep, query_idx_ep = [], []
            skip = False
            for c in classes_in_subj:
                c_mask = (y_train[subj_idx] == c)
                c_idx  = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx_ep.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx_ep.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx_ep = torch.cat(support_idx_ep)
            query_idx_ep   = torch.cat(query_idx_ep)

            # get support embeddings
            xs_time = X_time_train[support_idx_ep].to(device)
            xs_bp   = X_bp_train[support_idx_ep].to(device)
            xs_cov  = X_cov_train[support_idx_ep].to(device)
            xs_plv  = X_plv_train[support_idx_ep].to(device)
            xs_roi  = X_roi_train[support_idx_ep].to(device)
            ys      = y_train[support_idx_ep].to(device)

            # get query embeddings
            xq_time = X_time_train[query_idx_ep].to(device)
            xq_bp   = X_bp_train[query_idx_ep].to(device)
            xq_cov  = X_cov_train[query_idx_ep].to(device)
            xq_plv  = X_plv_train[query_idx_ep].to(device)
            xq_roi  = X_roi_train[query_idx_ep].to(device)
            yq      = y_train[query_idx_ep].to(device)

            model.train()
            e_support, _ = model(xs_time, xs_bp, xs_cov, xs_plv, xs_roi)
            e_query,   _ = model(xq_time, xq_bp, xq_cov, xq_plv, xq_roi)

            # compute prototypes from support
            classes_ep = torch.unique(ys)
            protos_ep  = torch.stack([
                e_support[ys == c].mean(dim=0) for c in classes_ep
            ])

            # classify query using prototypes (cosine similarity)
            e_query_norm  = F.normalize(e_query,   dim=1)
            protos_norm   = F.normalize(protos_ep, dim=1)
            logits_ep     = e_query_norm @ protos_norm.T / 0.1

            # map query labels to local indices
            label_map_ep = {int(c.item()): i for i, c in enumerate(classes_ep)}
            yq_local     = torch.tensor(
                [label_map_ep[int(v.item())] for v in yq],
                device=device
            )

            loss_episode = F.cross_entropy(logits_ep, yq_local)

            opt.zero_grad()
            loss_episode.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()
    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch", "encoder.bp_branch",
               "encoder.cov_branch",  "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            _ = model(
                xb_time.to(device), xb_bp.to(device),
                xb_cov.to(device),  xb_plv.to(device), xb_roi.to(device)
            )

    model.eval()
    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = F.normalize(all_e[mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~mask].to(device), dim=1)
    y_support = all_y[mask].to(device)
    y_query   = all_y[~mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred  = dists.argmin(dim=1)

    fewshot_acc = (pred == y_query).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nEPISODIC + CONTRASTIVE LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6698 | best 0.6698
Epoch 02 | zero-shot acc 0.6593 | best 0.6698
Epoch 03 | zero-shot acc 0.6246 | best 0.6698
Epoch 04 | zero-shot acc 0.6446 | best 0.6698
Epoch 05 | zero-shot acc 0.6519 | best 0.6698
Epoch 06 | zero-shot acc 0.6677 | best 0.6698
Epoch 07 | zero-shot acc 0.6193 | best 0.6698
Epoch 08 | zero-shot acc 0.6677 | best 0.6698
Epoch 09 | zero-shot acc 0.6562 | best 0.6698
Epoch 10 | zero-shot acc 0.6677 | best 0.6698
Epoch 11 | zero-shot acc 0.6635 | best 0.6698
Epoch 12 | zero-shot acc 0.6667 | best 0.6698
Epoch 13 | zero-shot acc 0.6225 | best 0.6698
Epoch 14 | zero-shot acc 0.6425 | best 0.6698
Epoch 15 | zero-shot acc 0.6341 | best 0.6698
AdaBN + Mahalanobis few-shot acc: 0.7845118045806885

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.7987 | best 0.7987
Epoch 02 | zero-shot acc 0.6741 | best 0.7987
Epoch 03 | zero-shot acc 0.6443 | best 0.7987
Epoch 04 | zero-shot acc 0.5325 | best 0.

In [ ]:
TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"


In [ ]:
torch.save({"model_state_dict": model.state_dict()}, TEACHER_CKPT_PATH)
print("Teacher saved to:", TEACHER_CKPT_PATH)

Teacher saved to: /content/drive/MyDrive/nback_teacher_best.pt


In [ ]:
# ============================================================
# 3-CHANNEL STUDENT TRAINING WITH EPISODIC + RKD DISTILLATION
# ============================================================

import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from collections import defaultdict

# ============================================================
# SETTINGS
# ============================================================

EPOCHS             = 15
BATCH              = 396
LR                 = 1e-3
LAMBDA             = 0.5
TEMP               = 0.1
PROTO_LAMBDA       = 0.10
FEW_SHOT_PER_CLASS = 20
SHRINK             = 0.4
N_EPISODES         = 100
DISTILL_LAMBDA     = 0.5

KEEP_IDXS = [0, 2, 13]   # AF3, F3, AF4
FS = 128
BANDS = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

TEACHER_CKPT_PATH = "/content/drive/MyDrive/nback_teacher_best.pt"

# ============================================================
# HELPERS
# ============================================================

class EEGDatasetStudent(Dataset):
    def __init__(self, X_time, X_bp, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )


class SubjectBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        shuffled = {s: inds[:] for s, inds in self.subj_to_indices.items()}
        for s in self.subjects:
            random.shuffle(shuffled[s])

        subj_ptr = {s: 0 for s in self.subjects}

        while True:
            if len(self.subjects) < self.subjects_per_batch:
                return

            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)
            batch = []

            for s in chosen_subjects:
                start = subj_ptr[s]
                end = start + self.samples_per_subject
                if end > len(shuffled[s]):
                    return
                batch.extend(shuffled[s][start:end])
                subj_ptr[s] = end

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return min_samples // self.samples_per_subject


def prototype_loss(emb, y):
    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]
        proto = class_emb.mean(dim=0)

        loss += ((class_emb - proto) ** 2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)


def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature
    sim = sim - sim.max(dim=1, keepdim=True).values

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    pos_counts = pos_mask.sum(dim=1)
    log_prob = sim - torch.log(denom)

    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


def prototype_ce_loss(e, y, temperature=0.1):
    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)

    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T
    logits = logits / temperature

    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y], device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss


def relational_distill_loss(e_student, e_teacher):
    s_sim = F.normalize(e_student, dim=1) @ F.normalize(e_student, dim=1).T
    t_sim = F.normalize(e_teacher, dim=1) @ F.normalize(e_teacher, dim=1).T
    return F.mse_loss(s_sim, t_sim)


def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    model.eval()

    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()


def compute_relative_bandpower_torch(X_time, fs=128, bands=(("theta", 4, 8), ("alpha", 8, 12), ("beta", 12, 30))):
    assert X_time.ndim == 3
    N, C, T = X_time.shape

    x_np = X_time.detach().cpu().numpy().astype(np.float32)

    freqs = np.fft.rfftfreq(T, d=1.0 / fs)
    X_fft = np.fft.rfft(x_np, axis=-1)
    PSD = (np.abs(X_fft) ** 2) / T

    bp_list = []
    for _, f_lo, f_hi in bands:
        mask = (freqs >= f_lo) & (freqs < f_hi)
        bp = PSD[..., mask].mean(axis=-1)
        bp_list.append(bp)

    X_bp = np.stack(bp_list, axis=-1)
    X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)
    X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

    return torch.tensor(X_bp_flat, dtype=torch.float32)


def extract_embeddings_student(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)


def extract_teacher_embeddings(teacher_model, X_time, X_bp, X_cov, X_plv, device, batch_size=256):
    teacher_model.eval()
    E = []

    X_roi = torch.zeros((X_bp.shape[0], 0), dtype=X_bp.dtype)

    with torch.no_grad():
        for i in range(0, len(X_time), batch_size):
            xb_time = X_time[i:i+batch_size].to(device)
            xb_bp   = X_bp[i:i+batch_size].to(device)
            xb_cov  = X_cov[i:i+batch_size].to(device)
            xb_plv  = X_plv[i:i+batch_size].to(device)
            xb_roi  = X_roi[i:i+batch_size].to(device)

            e, _ = teacher_model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)
            E.append(e.cpu())

    return torch.cat(E, dim=0)


# ============================================================
# STUDENT MODEL (3-CHANNEL)
# ============================================================

class FusionEncoderStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 3, kernel_size=7, padding=3, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 3, kernel_size=15, padding=7, groups=3),
            nn.BatchNorm1d(3),
            nn.GELU(),

            nn.Conv1d(3, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_roi):
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)

        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        z = torch.cat([z_time, z_bp], dim=1)
        e = self.embed(z)
        return e


class ContrastiveStudent3Ch(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0):
        super().__init__()
        self.encoder = FusionEncoderStudent3Ch(emb_dim=emb_dim, bp_dim=bp_dim, roi_dim=roi_dim)
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_roi):
        e = self.encoder(x_time, x_bp, x_roi)
        p = self.proj(e)
        return e, p


# ============================================================
# LOAD TEACHER
# ============================================================

teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
teacher_ckpt = torch.load(TEACHER_CKPT_PATH, map_location=device)

if isinstance(teacher_ckpt, dict) and "model_state_dict" in teacher_ckpt:
    teacher.load_state_dict(teacher_ckpt["model_state_dict"])
else:
    teacher.load_state_dict(teacher_ckpt)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Loaded teacher from:", TEACHER_CKPT_PATH)

# ============================================================
# BUILD 3-CHANNEL RAW TIME ONCE
# ============================================================

X_time_3_t = X_time_t[:, KEEP_IDXS, :].clone()
X_bp_3_t   = compute_relative_bandpower_torch(X_time_3_t, fs=FS, bands=BANDS)

print("3-ch time shape:", X_time_3_t.shape)
print("3-ch bp shape:  ", X_bp_3_t.shape)

# ============================================================
# LOSO TRAINING
# ============================================================

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "=" * 80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("=" * 80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx  = np.where(train_mask)[0]
    test_idx   = np.where(test_mask)[0]

    # =========================
    # RAW FEATURES
    # =========================
    X_time_train = X_time_3_t[train_idx].clone()
    X_time_test  = X_time_3_t[test_idx].clone()

    X_bp_train = X_bp_3_t[train_idx].clone()
    X_bp_test  = X_bp_3_t[test_idx].clone()

    X_time_train_teacher = X_time_t[train_idx].clone()
    X_time_test_teacher  = X_time_t[test_idx].clone()

    X_bp_train_teacher = X_bp_t[train_idx].clone()
    X_bp_test_teacher  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0, 2), keepdim=True)
    time_sd = X_time_train.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    time_mu_t = X_time_train_teacher.mean(dim=(0, 2), keepdim=True)
    time_sd_t = X_time_train_teacher.std(dim=(0, 2), keepdim=True) + 1e-6
    X_time_train_teacher = (X_time_train_teacher - time_mu_t) / time_sd_t
    X_time_test_teacher  = (X_time_test_teacher  - time_mu_t) / time_sd_t

    bp_mu_t = X_bp_train_teacher.mean(dim=0, keepdim=True)
    bp_sd_t = X_bp_train_teacher.std(dim=0, keepdim=True) + 1e-6
    X_bp_train_teacher = (X_bp_train_teacher - bp_mu_t) / bp_sd_t
    X_bp_test_teacher  = (X_bp_test_teacher  - bp_mu_t) / bp_sd_t

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0), dtype=X_bp_train.dtype)
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0), dtype=X_bp_test.dtype)

    # =========================
    # PRECOMPUTE TEACHER EMBEDDINGS
    # =========================
    teacher_e_train = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_train_teacher,
        X_bp=X_bp_train_teacher,
        X_cov=torch.zeros_like(X_cov_t[train_idx]),
        X_plv=X_plv_t[train_idx].clone(),
        device=device,
        batch_size=256
    )

    teacher_e_test = extract_teacher_embeddings(
        teacher_model=teacher,
        X_time=X_time_test_teacher,
        X_bp=X_bp_test_teacher,
        X_cov=torch.zeros_like(X_cov_t[test_idx]),
        X_plv=X_plv_t[test_idx].clone(),
        device=device,
        batch_size=256
    )

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDatasetStudent(X_time_train, X_bp_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDatasetStudent(X_time_test,  X_bp_test,  X_roi_test,  y_test,  subj_test)

    sampler      = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = ContrastiveStudent3Ch(emb_dim=64, proj_dim=64, bp_dim=9, roi_dim=0).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_acc   = -1.0
    best_state = None

    train_subj_tensor = train_ds.subj
    train_subjects    = torch.unique(train_subj_tensor).tolist()

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS + 1):
        student.train()

        # ── PART 1: regular contrastive training ──
        for batch_indices in train_loader.batch_sampler:
            xb_time = X_time_train[batch_indices].to(device)
            xb_bp   = X_bp_train[batch_indices].to(device)
            xb_roi  = X_roi_train[batch_indices].to(device)
            yb      = y_train[batch_indices].to(device)

            sb = train_ds.subj[batch_indices].to(device)
            teacher_batch = teacher_e_train[batch_indices].to(device)

            e, proj = student(xb_time, xb_bp, xb_roi)

            loss_metric  = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con     = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto   = prototype_loss(e, yb)
            loss_distill = relational_distill_loss(e, teacher_batch)

            loss = (
                loss_metric
                + LAMBDA     * loss_con
                + PROTO_LAMBDA * loss_proto
                + DISTILL_LAMBDA * loss_distill
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

        # ── PART 2: episodic few-shot training ──
        for episode in range(N_EPISODES):

            chosen_subj = random.choice(train_subjects)
            subj_mask   = (train_subj_tensor == chosen_subj)
            subj_idx    = subj_mask.nonzero(as_tuple=True)[0]

            classes_in_subj = torch.unique(y_train[subj_idx])
            if len(classes_in_subj) < 2:
                continue

            support_idx_ep, query_idx_ep = [], []
            skip = False
            for c in classes_in_subj:
                c_mask = (y_train[subj_idx] == c)
                c_idx  = subj_idx[c_mask]
                if len(c_idx) <= FEW_SHOT_PER_CLASS:
                    skip = True
                    break
                perm = torch.randperm(len(c_idx))
                support_idx_ep.append(c_idx[perm[:FEW_SHOT_PER_CLASS]])
                query_idx_ep.append(  c_idx[perm[FEW_SHOT_PER_CLASS:]])
            if skip:
                continue

            support_idx_ep = torch.cat(support_idx_ep)
            query_idx_ep   = torch.cat(query_idx_ep)

            xs_time = X_time_train[support_idx_ep].to(device)
            xs_bp   = X_bp_train[support_idx_ep].to(device)
            xs_roi  = X_roi_train[support_idx_ep].to(device)
            ys      = y_train[support_idx_ep].to(device)

            xq_time = X_time_train[query_idx_ep].to(device)
            xq_bp   = X_bp_train[query_idx_ep].to(device)
            xq_roi  = X_roi_train[query_idx_ep].to(device)
            yq      = y_train[query_idx_ep].to(device)

            student.train()
            e_support, _ = student(xs_time, xs_bp, xs_roi)
            e_query,   _ = student(xq_time, xq_bp, xq_roi)

            classes_ep = torch.unique(ys)
            protos_ep  = torch.stack([
                e_support[ys == c].mean(dim=0) for c in classes_ep
            ])

            e_query_norm = F.normalize(e_query,   dim=1)
            protos_norm  = F.normalize(protos_ep, dim=1)
            logits_ep    = e_query_norm @ protos_norm.T / 0.1

            label_map_ep = {int(c.item()): i for i, c in enumerate(classes_ep)}
            yq_local     = torch.tensor(
                [label_map_ep[int(v.item())] for v in yq],
                device=device
            )

            loss_episode = F.cross_entropy(logits_ep, yq_local)

            opt.zero_grad()
            loss_episode.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings_student(student, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos  = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits     = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map    = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    student.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)
    all_e_pre, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm      = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    support_mask = torch.zeros(len(all_y), dtype=torch.bool)
    support_mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in student.parameters():
        p.requires_grad = False

    student.eval()
    set_partial_bn_adapt(
        student,
        allow=("encoder.time_branch", "encoder.bp_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_roi  = xb_roi.to(device)
            _ = student(xb_time, xb_bp, xb_roi)

    student.eval()
    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = F.normalize(all_e[support_mask].to(device),  dim=1)
    e_query   = F.normalize(all_e[~support_mask].to(device), dim=1)
    y_support = all_y[support_mask].to(device)
    y_query   = all_y[~support_mask].to(device)

    var     = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()
    e_support = e_support * weights
    e_query   = e_query   * weights

    classes   = torch.unique(y_support)
    protos    = F.normalize(
        torch.stack([e_support[y_support == c].mean(dim=0) for c in classes]), dim=1
    )
    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0
    )

    D          = residuals.shape[1]
    cov        = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)
    avg_var    = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv    = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    pred  = dists.argmin(dim=1)

    label_map    = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor(
        [label_map[int(v.item())] for v in y_query], device=device
    )

    fewshot_acc = (pred == y_query_local).float().mean().item()
    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\n3-CH STUDENT + EPISODIC + RKD DISTILLATION LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))

Loaded teacher from: /content/drive/MyDrive/nback_teacher_best.pt
3-ch time shape: torch.Size([12159, 3, 512])
3-ch bp shape:   torch.Size([12159, 9])

FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6446 | best 0.6446
Epoch 02 | zero-shot acc 0.6193 | best 0.6446
Epoch 03 | zero-shot acc 0.6477 | best 0.6477
Epoch 04 | zero-shot acc 0.6625 | best 0.6625
Epoch 05 | zero-shot acc 0.6467 | best 0.6625
Epoch 06 | zero-shot acc 0.6435 | best 0.6625
Epoch 07 | zero-shot acc 0.6362 | best 0.6625
Epoch 08 | zero-shot acc 0.6299 | best 0.6625
Epoch 09 | zero-shot acc 0.6236 | best 0.6625
Epoch 10 | zero-shot acc 0.6309 | best 0.6625
Epoch 11 | zero-shot acc 0.6204 | best 0.6625
Epoch 12 | zero-shot acc 0.5857 | best 0.6625
Epoch 13 | zero-shot acc 0.5910 | best 0.6625
Epoch 14 | zero-shot acc 0.5710 | best 0.6625
Epoch 15 | zero-shot acc 0.7087 | best 0.7087
AdaBN + Mahalanobis few-shot acc: 0.9393939971923828

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3

KeyboardInterrupt: 